In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from datetime import datetime
import warnings
import re
from collections import Counter
import string
import pickle
import scipy.sparse as sp
import os

import spacy
import nltk
from nltk.corpus import stopwords
from tqdm.notebook import tqdm

from sentence_transformers import SentenceTransformer

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

In [ ]:
DATA_PATH = '../'
original_data = pd.read_csv(f'{DATA_PATH}rating.csv')
print(f'Original shape: {original_data.shape[0]:,} rows, {original_data.shape[1]} columns')

In [ ]:
df = original_data[['title', 'title_sentiment', 'source_name', 'published_at']].copy()
df = df.drop_duplicates(subset=['title'], keep='first')
df = df.dropna(subset=['title', 'title_sentiment'])
print(f'After cleaning: {len(df):,}')

In [ ]:
print('Sentiment distribution (before balancing):')
print(df['title_sentiment'].value_counts())

In [ ]:
sentiment_counts = df['title_sentiment'].value_counts()
colors = {'Positive': '#2ecc71', 'Neutral': '#3498db', 'Negative': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax1 = axes[0]
bars = ax1.bar(sentiment_counts.index, sentiment_counts.values, 
               color=[colors.get(x, '#95a5a6') for x in sentiment_counts.index])
ax1.set_title('Sentiment Distribution (Counts)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Sentiment')
ax1.set_ylabel('Number of Articles')

ax2 = axes[1]
ax2.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%',
        colors=[colors.get(x, '#95a5a6') for x in sentiment_counts.index])
ax2.set_title('Sentiment Distribution (%)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top_sources = df['source_name'].value_counts().head(15)

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(top_sources.index[::-1], top_sources.values[::-1], color='steelblue')
ax.set_title('Top 15 News Sources', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Articles')
plt.tight_layout()
plt.savefig('../outputs/top_sources.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
top_sources_list = df['source_name'].value_counts().head(10).index.tolist()
source_sentiment = df[df['source_name'].isin(top_sources_list)].groupby(
    ['source_name', 'title_sentiment']).size().unstack(fill_value=0)
source_sentiment_pct = source_sentiment.div(source_sentiment.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(12, 6))
source_sentiment_pct.plot(kind='barh', stacked=True, ax=ax, 
                          color=[colors['Negative'], colors['Neutral'], colors['Positive']])
ax.set_title('Sentiment Distribution by Source (Top 10)', fontsize=14, fontweight='bold')
ax.set_xlabel('Percentage (%)')
ax.set_ylabel('Source')
ax.legend(title='Sentiment', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.savefig('../outputs/sentiment_by_source.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['published_at'] = pd.to_datetime(df['published_at'], errors='coerce')
df_temp = df.dropna(subset=['published_at']).copy()
df_temp['date'] = df_temp['published_at'].dt.date
df_temp['week'] = df_temp['published_at'].dt.to_period('W').astype(str)

weekly_counts = df_temp.groupby('week').size()
weekly_sentiment = df_temp.groupby(['week', 'title_sentiment']).size().unstack(fill_value=0)
weekly_sentiment_pct = weekly_sentiment.div(weekly_sentiment.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(2, 1, figsize=(14, 10))

axes[0].bar(range(len(weekly_counts)), weekly_counts.values, color='steelblue')
axes[0].set_title('Weekly Publication Volume', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Week')
axes[0].set_ylabel('Number of Articles')
axes[0].set_xticks(range(0, len(weekly_counts), max(1, len(weekly_counts)//10)))
axes[0].set_xticklabels([weekly_counts.index[i] for i in range(0, len(weekly_counts), max(1, len(weekly_counts)//10))], rotation=45)

weekly_sentiment_pct.plot(kind='bar', stacked=True, ax=axes[1], 
                          color=[colors['Negative'], colors['Neutral'], colors['Positive']], width=0.8)
axes[1].set_title('Weekly Sentiment Distribution (%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Week')
axes[1].set_ylabel('Percentage (%)')
axes[1].legend(title='Sentiment')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/temporal_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df['title_length'] = df['title'].str.len()
df['word_count'] = df['title'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sentiment in ['Positive', 'Neutral', 'Negative']:
    data = df[df['title_sentiment'] == sentiment]['title_length']
    axes[0].hist(data, bins=50, alpha=0.5, label=sentiment, color=colors[sentiment])
axes[0].set_title('Title Length Distribution by Sentiment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Character Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

word_count_by_sentiment = df.groupby('title_sentiment')['word_count'].mean()
word_count_by_sentiment = word_count_by_sentiment.reindex(['Negative', 'Neutral', 'Positive'])
axes[1].bar(word_count_by_sentiment.index, word_count_by_sentiment.values,
           color=[colors[x] for x in word_count_by_sentiment.index])
axes[1].set_title('Average Word Count by Sentiment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sentiment')
axes[1].set_ylabel('Word Count')

plt.tight_layout()
plt.savefig('../outputs/text_length_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
stop_words = set(stopwords.words('english'))

def get_word_freq(texts, n=20):
    all_words = []
    for text in texts:
        if pd.isna(text):
            continue
        words = str(text).lower().split()
        words = [w.strip(string.punctuation) for w in words if w.strip(string.punctuation) and w not in stop_words]
        all_words.extend(words)
    return Counter(all_words).most_common(n)

all_freq = get_word_freq(df['title'], 20)

fig, ax = plt.subplots(figsize=(12, 6))
words = [w[0] for w in all_freq][::-1]
counts = [w[1] for w in all_freq][::-1]
ax.barh(words, counts, color='steelblue')
ax.set_title('Top 20 Words in Headlines', fontsize=14, fontweight='bold')
ax.set_xlabel('Frequency')
plt.tight_layout()
plt.savefig('../outputs/word_frequency.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, sentiment in enumerate(['Negative', 'Neutral', 'Positive']):
    sent_texts = df[df['title_sentiment'] == sentiment]['title']
    freq = get_word_freq(sent_texts, 15)
    words = [w[0] for w in freq][::-1]
    counts = [w[1] for w in freq][::-1]
    axes[idx].barh(words, counts, color=colors[sentiment])
    axes[idx].set_title(f'Top Words: {sentiment}', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Frequency')

plt.tight_layout()
plt.savefig('../outputs/words_by_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
if 'category' in original_data.columns:
    cat_counts = original_data['category'].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(cat_counts.index[::-1], cat_counts.values[::-1], color='coral')
    ax.set_title('Top 15 Categories', fontsize=14, fontweight='bold')
    ax.set_xlabel('Number of Articles')
    plt.tight_layout()
    plt.savefig('../outputs/category_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
negative = df[df['title_sentiment'] == 'Negative']
positive = df[df['title_sentiment'] == 'Positive']
neutral = df[df['title_sentiment'] == 'Neutral']

neutral_target_size = len(negative) * 2
neutral_subset = neutral.sample(n=min(neutral_target_size, len(neutral)), random_state=42)

df_balanced = pd.concat([positive, negative, neutral_subset], ignore_index=True)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'Balanced dataset: {len(df_balanced):,}')
print(df_balanced['title_sentiment'].value_counts())

In [ ]:
def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'&[a-zA-Z]+;', '', text)
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    text = emoji_pattern.sub('', text)
    return text.strip()

def preprocess_text_nlp(text):
    nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
    stop_words = set(stopwords.words('english'))
    
    if pd.isna(text) or text == '':
        return ''
    
    text = clean_text(text)
    text = text.lower()
    
    doc = nlp(text)
    
    tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        if token.text in stop_words:
            continue
        if token.like_num:
            continue
        lemma = token.lemma_.strip()
        if len(lemma) > 1:
            tokens.append(lemma)
    
    return ' '.join(tokens)

In [ ]:
tqdm.pandas(desc='Preprocessing')
df_balanced['title_processed'] = df_balanced['title'].progress_apply(preprocess_text_nlp)
df_balanced = df_balanced[df_balanced['title_processed'].str.len() > 0].reset_index(drop=True)
print(f'After preprocessing: {len(df_balanced):,}')

In [ ]:
sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
sentiment_mapping_reverse = {v: k for k, v in sentiment_mapping.items()}
df_balanced['sentiment_label'] = df_balanced['title_sentiment'].map(sentiment_mapping)

In [ ]:
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(df_balanced['title_processed'])
y = df_balanced['sentiment_label'].values

print(f'TF-IDF matrix shape: {X_tfidf.shape}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
sp.save_npz('../outputs/X_train_tfidf.npz', X_train)
sp.save_npz('../outputs/X_test_tfidf.npz', X_test)
np.save('../outputs/y_train.npy', y_train)
np.save('../outputs/y_test.npy', y_test)

with open('../models/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

with open('../models/sentiment_mapping.pkl', 'wb') as f:
    pickle.dump({'forward': sentiment_mapping, 'reverse': sentiment_mapping_reverse}, f)

df_balanced.to_csv('../outputs/df_processed.csv', index=False)

In [ ]:
model = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    solver='lbfgs',
    multi_class='multinomial',
    n_jobs=-1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 (macro): {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'F1 (weighted): {f1_score(y_test, y_pred, average="weighted"):.4f}')

In [ ]:
target_names = [sentiment_mapping_reverse[i] for i in sorted(sentiment_mapping_reverse.keys())]
print(classification_report(y_test, y_pred, target_names=target_names, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test, y_pred)
disp1 = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')

cm_norm = confusion_matrix(y_test, y_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(cm_norm, display_labels=target_names)
disp2.plot(ax=axes[1], cmap='Blues', values_format='.2%')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/confusion_matrix_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_full = sp.vstack([X_train, X_test])
y_full = np.concatenate([y_train, y_test])

cv_accuracy = cross_val_score(model, X_full, y_full, cv=cv, scoring='accuracy', n_jobs=-1)
cv_f1_macro = cross_val_score(model, X_full, y_full, cv=cv, scoring='f1_macro', n_jobs=-1)

print(f'CV Accuracy: {cv_accuracy.mean():.4f} (+/- {cv_accuracy.std()*2:.4f})')
print(f'CV F1 (macro): {cv_f1_macro.mean():.4f} (+/- {cv_f1_macro.std()*2:.4f})')

In [ ]:
feature_names = tfidf.get_feature_names_out()
coef = model.coef_
n_top = 20

fig, axes = plt.subplots(1, 3, figsize=(18, 8))

for idx, (label, name) in enumerate(sentiment_mapping_reverse.items()):
    top_indices = np.argsort(coef[label])[-n_top:][::-1]
    top_features = [(feature_names[i], coef[label][i]) for i in top_indices]
    
    ax = axes[idx]
    words = [f[0] for f in top_features][::-1]
    weights = [f[1] for f in top_features][::-1]
    
    ax.barh(words, weights, color=colors[name])
    ax.set_title(f'Top {n_top} Features: {name}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/feature_importance_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
with open('../models/logistic_regression_baseline.pkl', 'wb') as f:
    pickle.dump(model, f)

In [ ]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')
titles = df_balanced['title'].fillna('').tolist()
embeddings = encoder.encode(titles, show_progress_bar=True, batch_size=64, convert_to_numpy=True)
print(f'Embeddings shape: {embeddings.shape}')

In [ ]:
np.save('../outputs/embeddings_minilm.npy', embeddings)

In [ ]:
X_emb = embeddings
y_emb = df_balanced['sentiment_label'].values

X_train_emb, X_test_emb, y_train_emb, y_test_emb = train_test_split(
    X_emb, y_emb, test_size=0.2, random_state=42, stratify=y_emb
)

scaler = StandardScaler()
X_train_emb_scaled = scaler.fit_transform(X_train_emb)
X_test_emb_scaled = scaler.transform(X_test_emb)

with open('../models/embedding_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [ ]:
models_dict = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42, n_jobs=-1)
}

results = []

for name, clf in models_dict.items():
    clf.fit(X_train_emb, y_train_emb)
    y_pred_emb = clf.predict(X_test_emb)
    
    acc = accuracy_score(y_test_emb, y_pred_emb)
    f1_mac = f1_score(y_test_emb, y_pred_emb, average='macro')
    
    results.append({'Model': name, 'Accuracy': acc, 'F1 (macro)': f1_mac})
    print(f'{name}: Acc {acc:.4f}, F1 {f1_mac:.4f}')

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
best_model_emb = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42, n_jobs=-1)
best_model_emb.fit(X_train_emb, y_train_emb)
y_pred_emb = best_model_emb.predict(X_test_emb)

print(classification_report(y_test_emb, y_pred_emb, target_names=target_names, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(y_test_emb, y_pred_emb)
disp1 = ConfusionMatrixDisplay(cm, display_labels=target_names)
disp1.plot(ax=axes[0], cmap='Blues', values_format='d')
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')

cm_norm = confusion_matrix(y_test_emb, y_pred_emb, normalize='true')
disp2 = ConfusionMatrixDisplay(cm_norm, display_labels=target_names)
disp2.plot(ax=axes[1], cmap='Blues', values_format='.2%')
axes[1].set_title('Confusion Matrix (Normalized)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../outputs/confusion_matrix_embeddings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sentiment_embeddings = {}
for label, name in sentiment_mapping_reverse.items():
    mask = y_emb == label
    sentiment_embeddings[name] = embeddings[mask].mean(axis=0)

names = list(sentiment_embeddings.keys())
vectors = np.array([sentiment_embeddings[n] for n in names])
sim_matrix = cosine_similarity(vectors)

sim_df = pd.DataFrame(sim_matrix, index=names, columns=names)
print(sim_df.round(4))

In [ ]:
with open('../models/logreg_embeddings.pkl', 'wb') as f:
    pickle.dump(best_model_emb, f)

np.save('../outputs/X_train_embeddings.npy', X_train_emb)
np.save('../outputs/X_test_embeddings.npy', X_test_emb)

In [ ]:
print(f'TF-IDF Baseline: Acc {accuracy_score(y_test, y_pred):.4f}, F1 {f1_score(y_test, y_pred, average="macro"):.4f}')
print(f'Embeddings: Acc {accuracy_score(y_test_emb, y_pred_emb):.4f}, F1 {f1_score(y_test_emb, y_pred_emb, average="macro"):.4f}')